# YZM 513 — Week 6 Lab: Train One Neural Network, From NumPy to the GPU

**İstanbul Medeniyet Üniversitesi — AI Engineering M.Sc. (Fall 2026)**

We train **one** small neural network (an MLP that reads handwritten digits, MNIST) and carry it through the whole course so far. This notebook is also the **review of Weeks 1–6** before the midterm:

| Step | What we do | Course week it reviews |
|---|---|---|
| **A** | Warm-up on the CPU: measure, vectorise, broadcast, BLAS threads | Weeks 1–4 |
| **B** | Write the MLP *as matrices*; check the gradients; train it with NumPy | Weeks 1, 2, 5 |
| **C** | Meet the GPU with simple array tasks (CuPy) | **Week 6** |
| **D** | Run the **same** training code on the GPU; find when the GPU wins | **Week 6** |
| **E** | Same model in PyTorch (autograd) and a mixed-precision preview | Weeks 9–12 preview |
| **F** | Summary table and your exercises | |

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (about 5–8 minutes).
Cells marked 📝 ask you to write something down. Cells marked 🔧 are your turn.

> **Why this example?** An MLP on MNIST is small enough for a free T4 and for you to check by hand,
> yet *every* line of it is a matrix operation you already met in Weeks 1–5. Nothing is hidden inside
> a framework until Part E, so you can see exactly *what* the GPU is speeding up.

### Learning goals

By the end, you should be able to explain **not just which backend was faster**, but also:

- where the data and intermediate arrays live;
- why a GPU timing without synchronisation is invalid;
- why transfers, launch overhead, dtype, batch size, and model width change the speedup;
- why NumPy, CuPy, and PyTorch can produce similar accuracy without producing bit-identical weights.

### How to work through the notebook

1. **Run cells in order.** Later parts reuse functions and variables defined earlier.
2. **Predict before measuring.** Where a cell asks for a prediction, write it down before running the next cell.
3. **Record trends, not just digits.** Timings vary with the machine, GPU clocks, other load, and warm-up state.
4. If you change a model or training setting, **rerun the relevant baseline** so the comparison remains fair.

Throughout, `xp` means “the array module”: it is `np` for NumPy or `cp` for CuPy. `B` is batch size, `H` is hidden width, and `epoch_s` is training time per epoch unless a cell says otherwise.


## 0. Setup

First check that Colab gave you a GPU. If `nvidia-smi` fails, switch the runtime type to **T4 GPU** and start again.

In the device-information output, check four things:

- **GPU name:** this identifies the hardware you are actually benchmarking.
- **SM count:** streaming multiprocessors are the hardware units that execute groups of GPU threads.
- **Memory:** later, an out-of-memory error means the model, activations, or gradients no longer fit—not that the GPU is “too slow”.
- **CPU cores:** the NumPy baseline depends strongly on the CPU and its BLAS configuration, so do not transfer the measured speedup to another machine without rerunning it.


In [ ]:
# On Colab, select Runtime → Change runtime type → T4 GPU before running this cell.
!nvidia-smi


In [ ]:
# Install the CuPy build that matches Colab's CUDA driver (Colab usually has CuPy already).
import re, subprocess, sys
try:
    import cupy as cp
except ImportError:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    m = re.search(r"CUDA Version:\s*(\d+)", smi)
    assert m, "No NVIDIA GPU found. Runtime → Change runtime type → T4 GPU, then run again."
    pkg = f"cupy-cuda{m.group(1)}x"      # e.g. cupy-cuda12x
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    import cupy as cp

In [ ]:
import os, time, math, urllib.request
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

assert cp.cuda.runtime.getDeviceCount() > 0, "No GPU visible: switch the runtime to T4 GPU."

dev   = cp.cuda.Device(0)
props = cp.cuda.runtime.getDeviceProperties(0)
name  = props.get("name", props.get(b"name"))
free, total = dev.mem_info
print("CuPy        :", cp.__version__ if hasattr(cp, "__version__") else "?", "| CUDA runtime", cp.cuda.runtime.runtimeGetVersion())
print("GPU         :", name.decode() if isinstance(name, bytes) else name)
print("SMs         :", dev.attributes["MultiProcessorCount"], "| compute capability", dev.compute_capability)
print(f"GPU memory  : {total/2**30:.1f} GiB total, {free/2**30:.1f} GiB free")
print("CPU cores   :", os.cpu_count(), "(Colab gives you only a couple, so our NumPy baseline is a *modest* CPU)")

### Two helpers we will use all day

* `timeit(fn, sync=...)`: run a function several times and report the **median** time.
* `sync_gpu()`: wait until the GPU has *really* finished. GPU calls are **asynchronous**: Python continues as soon as the work is *queued*. Without a sync, you time the queueing, not the work (we prove this in Part C).

Two small details make the measurements more trustworthy:

- **Warm-up** excludes one-time costs such as kernel compilation and library initialisation from the steady-state timing.
- The **median** reduces the effect of one unusually slow repetition. It does not remove systematic problems such as forgotten transfers.

For CPU work, no synchronisation helper is needed. For GPU work, the helper waits on the default stream so that the wall clock includes execution as well as Python's launch time.


In [ ]:
def sync_gpu():
    cp.cuda.Stream.null.synchronize()

def timeit(fn, repeat=5, warmup=1, sync=None):
    for _ in range(warmup):
        fn()
    if sync: sync()
    ts = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        if sync: sync()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

results = {}   # we fill this in as we go: backend -> dict(epoch_s, acc)

---
# Part A — Warm-up on the CPU (Weeks 1–4 in 15 minutes)

Before any GPU, let's re-check the habits that will carry over. Everything here runs on the CPU with NumPy.

## A1 · Week 1: *measure first*. How many FLOPs is one training step?

A layer `Z = H @ W + b` with a batch of `B` rows, `n_in` inputs and `n_out` outputs costs `2·B·n_in·n_out` FLOPs
(one multiply + one add per term). A training step is a forward pass plus a backward pass that needs about **two more** matmuls per layer,
so one step ≈ **3 × forward**.

Be careful with the units:

- **FLOP** is an amount of work: floating-point operations.
- **FLOP/s** is a rate: how quickly the hardware completes that work.
- “2048² matmul” is shorthand for multiplying two matrices of shape **2048 × 2048**.

The FLOP calculation is an estimate of the dense linear algebra. It does not mean every operation in a real training step is a matmul, and it ignores effects such as memory traffic and launch overhead.


In [ ]:
def step_flops(B, sizes):
    fwd = sum(2 * B * a * b for a, b in zip(sizes[:-1], sizes[1:]))
    return 3 * fwd

B, sizes = 256, [784, 512, 10]
fl = step_flops(B, sizes)
print(f"one training step ≈ {fl/1e6:.0f} MFLOP;  one epoch (60,000 rows = {60000//B} steps) ≈ {fl*(60000//B)/1e9:.0f} GFLOP")

# How fast is this machine's NumPy at its best?  (a big float32 matmul)
a = np.random.rand(2048, 2048).astype(np.float32)
t = timeit(lambda: a @ a)
print(f"NumPy matmul 2048²: {t*1e3:.0f} ms  →  {2*2048**3/t/1e9:.0f} GFLOP/s  (this is the CPU's practical peak for us)")

### How to read A1

The first line estimates work per step and per epoch. The second line measures a large, well-tuned CPU matmul and converts its time into GFLOP/s:

$$\text{achieved GFLOP/s}=\frac{2n^3}{t\times 10^9}.$$

We call this our **practical CPU peak** because it is a useful yardstick from a real NumPy operation, not a guaranteed maximum for every workload. A small training step may achieve much less because it launches many operations and moves smaller arrays.


📝 *Write down:* CPU practical peak = ______ GFLOP/s. A T4 advertises ~8,000 GFLOP/s (FP32). Keep this number: in Part C we measure how much of that we actually get.

## A2 · Week 2: *vectorise*. Loops vs NumPy vs Numba

The same `C = A @ B` three ways. (Pure Python is tiny here, because it is *that* slow.)

All three implementations compute the same mathematical product, but they execute it very differently:

- **Pure Python** interprets every loop iteration and arithmetic operation.
- **Numba** compiles the loops to machine code; the first call includes compilation, so we compile before timing.
- **NumPy `@`** calls a tuned BLAS library that uses optimised loops, cache-friendly tiling, vector instructions, and usually multiple CPU cores.

The small example makes the Python loop manageable; the larger example makes the difference between compiled loops and tuned BLAS clearer.


In [ ]:
from numba import njit

def matmul_python(A, B):
    n, m, p = len(A), len(B), len(B[0])
    C = [[0.0] * p for _ in range(n)]
    for i in range(n):
        for k in range(m):
            for j in range(p):
                C[i][j] += A[i][k] * B[k][j]
    return C

@njit(fastmath=True)
def matmul_numba(A, B):
    n, m = A.shape; p = B.shape[1]
    C = np.zeros((n, p), dtype=A.dtype)
    for i in range(n):
        for k in range(m):
            for j in range(p):
                C[i, j] += A[i, k] * B[k, j]
    return C

n = 120
A = np.random.rand(n, n).astype(np.float32); Bm = np.random.rand(n, n).astype(np.float32)
t_py = timeit(lambda: matmul_python(A.tolist(), Bm.tolist()), repeat=1, warmup=0)
matmul_numba(A, Bm)                         # first call compiles
t_nb = timeit(lambda: matmul_numba(A, Bm))
t_np = timeit(lambda: A @ Bm)
print(f"n={n}:  pure Python {t_py*1e3:8.1f} ms | Numba loop {t_nb*1e3:7.3f} ms | NumPy @ {t_np*1e3:7.3f} ms")

n = 1024
A = np.random.rand(n, n).astype(np.float32); Bm = np.random.rand(n, n).astype(np.float32)
t_nb = timeit(lambda: matmul_numba(A, Bm), repeat=3)
t_np = timeit(lambda: A @ Bm, repeat=3)
print(f"n={n}: Numba loop {t_nb*1e3:7.1f} ms | NumPy @ {t_np*1e3:7.1f} ms   → tuned BLAS is {t_nb/t_np:.0f}× faster than our own loop")

**What to notice:** Numba can make a loop dramatically faster without changing its structure, but a specialised library can still win by optimising data movement and parallelism as well as arithmetic. This is why “compiled” does not automatically mean “fastest possible”.


**Take-away:** the Python loop is hopeless, Numba rescues it, and the vendor-tuned BLAS (`@`) still wins. On the GPU the same story repeats: **cuBLAS** is the tuned matmul, and CuPy's `@` calls it.

## A3 · Week 2: broadcasting and a *stable* softmax

Our network ends with a softmax. The naive formula overflows; subtracting the row maximum fixes it. This is broadcasting: `(B, 10) - (B, 1)`.

Subtracting the row maximum does not change the mathematical softmax:

$$\operatorname{softmax}(z)_i=\frac{e^{z_i}}{\sum_j e^{z_j}}
=\frac{e^{z_i-m}}{\sum_j e^{z_j-m}},\qquad m=\max_j z_j.$$

It changes the **numerical range** of the exponentials. After subtraction, the largest logit is zero, so its exponential is one rather than infinity. `keepdims=True` preserves the row dimension so NumPy can broadcast one maximum over each row.


In [ ]:
logits = np.array([[1000., 1001., 1002.]], dtype=np.float32)

naive = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)       # exp(1000) = inf
print("naive :", naive)

z = logits - logits.max(axis=1, keepdims=True)                           # (1,3) - (1,1) → broadcast
stable = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
print("stable:", stable, "| row sum =", stable.sum())

Check two things in the output: the naive calculation contains `nan`, while the stable calculation contains finite probabilities whose rows sum to one (up to floating-point rounding). The largest logit should receive the largest probability in both the mathematical and stable computations.

**additional reading:** Barron, Jonathan T., Dan Gnanapragasam, and Michael Tschannen. "A Fix for Softmax and Cross-Entropy." arXiv preprint arXiv:2610.31642 (2026). [pdf](https://arxiv.org/pdf/2610.31642)

## A4 · Weeks 3–4: do we need threads or processes for the matmul?

No. `A @ B` already runs on **all your cores inside the BLAS library** and does not hold the GIL (Week 4).
Let's see the threads at work by limiting BLAS to one thread.

Adding a Python thread or process pool **around** an already parallel BLAS matmul can create **oversubscription**: several workers compete for the same cores and memory bandwidth. Threads and processes are more useful around the matmul—for data loading, augmentation, tokenisation, or independent tasks.


In [ ]:
try:
    from threadpoolctl import threadpool_limits
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "threadpoolctl"], check=True)
    from threadpoolctl import threadpool_limits

a = np.random.rand(2048, 2048).astype(np.float32)
for k in sorted({1, os.cpu_count()}):
    with threadpool_limits(limits=k, user_api="blas"):
        t = timeit(lambda: a @ a)
    print(f"BLAS threads = {k}:  {t*1e3:7.1f} ms   ({2*2048**3/t/1e9:5.0f} GFLOP/s)")

The timing should improve when BLAS is allowed to use more than one thread, but the scaling will not necessarily equal the core count. Matmul also depends on caches and memory bandwidth, and the runtime may not give you exclusive access to all reported CPUs.

The GPU comparison is conceptually similar—more parallel arithmetic becomes available—but a GPU “CUDA core” is not equivalent to an independent CPU core. We will therefore compare **measured workload performance**, not just core counts.


---
# Part B — The MLP written as matrices (CPU, NumPy)

## B1 · Data = a matrix

MNIST: 60,000 training images of 28×28 pixels, flattened to rows of 784 numbers, with a label 0–9.
We keep everything **`float32`** (a deliberate habit: GPUs are built for 32-bit floats, and NumPy silently produces `float64`).

Normalisation uses broadcasting in two different shapes:

- `mu` has shape `(784,)`: one mean per pixel, applied to every image;
- `sd` is one scalar standard deviation, applied to every value.

Both statistics are computed from the **training set only** and then applied unchanged to the test set. Using test-set statistics would leak information from the evaluation data into preprocessing.


In [ ]:
def load_mnist():
    path = "mnist.npz"
    if not os.path.exists(path):
        urllib.request.urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", path)
    d = np.load(path)
    return (d["x_train"].reshape(-1, 784), d["y_train"].astype(np.int64),
            d["x_test"].reshape(-1, 784),  d["y_test"].astype(np.int64))

def fake_digits(n_train=60000, n_test=10000, seed=0):
    # Offline fallback: 10 noisy "prototype" images. Same shapes as MNIST, easier to learn.
    rng = np.random.default_rng(seed)
    protos = rng.random((10, 784)) * 255
    def make(n):
        y = rng.integers(0, 10, n)
        return np.clip(protos[y] + rng.normal(0, 90, (n, 784)), 0, 255), y.astype(np.int64)
    (Xa, ya), (Xb, yb) = make(n_train), make(n_test)
    return Xa, ya, Xb, yb

try:
    Xtr, ytr, Xte, yte = load_mnist(); source = "MNIST"
except Exception as e:
    print("MNIST download failed (", type(e).__name__, ") → using synthetic digits");
    Xtr, ytr, Xte, yte = fake_digits(); source = "synthetic"

Xtr = Xtr.astype(np.float32) / 255; Xte = Xte.astype(np.float32) / 255
mu, sd = Xtr.mean(axis=0), Xtr.std()          # (784,) per-pixel mean, one global std → broadcasting
Xtr = (Xtr - mu) / sd;  Xte = (Xte - mu) / sd
data = (Xtr, ytr, Xte, yte)
print(source, "| X_train", Xtr.shape, Xtr.dtype, f"({Xtr.nbytes/1e6:.0f} MB)", "| y_train", ytr.shape, "| X_test", Xte.shape)

### Data checkpoint

Before training, verify that `X_train` has 60,000 rows and 784 columns, that the features are `float32`, and that the labels remain integer class indices. Each row of `Xtr` is one image; each column is one pixel position.

The printed byte count is the raw feature array only. Labels, test data, model parameters, activations, and gradients require additional memory.

> If the notebook reports **synthetic** data, the code and timing experiments still work, but the accuracy is not MNIST accuracy. Do not compare it with published MNIST results; restore internet access and rerun the data cell for the intended experiment.


In [ ]:
fig, axs = plt.subplots(1, 8, figsize=(12, 1.8))
for i, ax in enumerate(axs):
    ax.imshow((Xtr[i] * sd + mu).reshape(28, 28), cmap="gray"); ax.set_title(f"label {ytr[i]}"); ax.axis("off")
plt.show()

## B2 · The model, in four formulas

For a batch `X` (B×784), hidden width `H`, weights `W1` (784×H), `W2` (H×10):

$$H_1=\max(0,\;XW_1+b_1)\qquad L=H_1W_2+b_2\qquad P=\mathrm{softmax}(L)\qquad \mathcal{L}=-\tfrac1B\sum_i\log P_{i,y_i}$$

and, by the chain rule, with `dL = (P − onehot(y)) / B`:

$$\frac{\partial\mathcal L}{\partial W_2}=H_1^{\top}dL\qquad \frac{\partial\mathcal L}{\partial b_2}=\sum_{\text{rows}} dL\qquad dH_1=(dL\,W_2^{\top})\odot[H_1>0]\qquad \frac{\partial\mathcal L}{\partial W_1}=X^{\top}dH_1$$

Count the matmuls: 2 forward, 3 backward (the 5th, `dL·W2ᵀ`, is needed to continue backwards). **Everything else is cheap element-wise work.**

### The one trick that makes the GPU move painless

We write each function so it receives the array module `xp` (either `numpy` or `cupy`).
The code is identical; only the arrays live somewhere else.

The parameter list is ordered as `[W1, b1, W2, b2]`. For a two-layer network with batch size `B` and hidden width `H`, the important shapes are:

| Quantity | Shape | Meaning |
|---|---|---|
| `X` | `(B, 784)` | one flattened image per row |
| `W1`, `b1` | `(784, H)`, `(H,)` | input → hidden weights and bias |
| hidden activation | `(B, H)` | one hidden vector per image |
| `W2`, `b2` | `(H, 10)`, `(10,)` | hidden → class-score weights and bias |
| logits, probabilities | `(B, 10)` | ten scores/probabilities per image |

The forward pass saves intermediate activations because backpropagation needs them. The backward pass then propagates derivatives in reverse order: output gradients → hidden gradients → parameter gradients.


In [ ]:
def init_params(sizes, seed=0, dtype=np.float32):
    # He initialisation, always generated on the CPU so every backend starts from identical weights.
    rng = np.random.default_rng(seed)
    params = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        params.append((rng.standard_normal((n_in, n_out)) * math.sqrt(2.0 / n_in)).astype(dtype))
        params.append(np.zeros(n_out, dtype=dtype))
    return params

def forward(xp, params, X):
    acts, h = [X], X
    n_layers = len(params) // 2
    for l in range(n_layers):
        z = h @ params[2*l] + params[2*l + 1]          # matmul + broadcast bias
        h = xp.maximum(z, 0) if l < n_layers - 1 else z  # ReLU on hidden layers, raw logits at the end
        acts.append(h)
    return acts                                        # acts[-1] are the logits

def softmax_xent(xp, logits, y):
    z = logits - logits.max(axis=1, keepdims=True)
    logp = z - xp.log(xp.exp(z).sum(axis=1, keepdims=True))
    B = logits.shape[0]
    rows = xp.arange(B)
    loss = -logp[rows, y].mean()
    dlogits = xp.exp(logp)                             # softmax probabilities
    dlogits[rows, y] -= 1.0                            # P - onehot(y)
    return loss, dlogits / B

def backward(xp, params, acts, dlogits):
    grads = [None] * len(params)
    d = dlogits
    for l in reversed(range(len(params) // 2)):
        grads[2*l]     = acts[l].T @ d
        grads[2*l + 1] = d.sum(axis=0)
        if l > 0:
            d = (d @ params[2*l].T) * (acts[l] > 0)    # through W, then through the ReLU
    return grads

def loss_and_grads(xp, params, X, y):
    acts = forward(xp, params, X)
    loss, dl = softmax_xent(xp, acts[-1], y)
    return loss, backward(xp, params, acts, dl)

def train_step(xp, params, X, y, lr):
    loss, grads = loss_and_grads(xp, params, X, y)
    for p, g in zip(params, grads):
        p -= lr * g                                    # in-place SGD update
    return loss

### How to read the model code

Follow one batch through these functions rather than trying to read the whole cell at once:

1. `forward` computes each layer and returns `acts`; `acts[0]` is the input and `acts[-1]` contains the logits.
2. `softmax_xent` returns both the mean cross-entropy loss and `dlogits`, the derivative with respect to the logits.
3. `backward` walks through the layers in reverse and returns gradients in the same `[W1, b1, W2, b2]` order as `params`.
4. `train_step` performs forward → backward → one in-place SGD update.
5. Passing `np` or `cp` as `xp` changes the array backend without changing these steps.

For the output layer, the combined softmax–cross-entropy derivative simplifies to `(P − onehot(y)) / B`. That is why the code does not separately construct the full Jacobian of softmax.


## B3 · Correct first, fast second: gradient check

A hand-written backward pass is easy to get wrong, and a wrong model on a fast GPU is just a *fast wrong model*.
We compare our gradients with finite differences, $\frac{L(w+\varepsilon)-L(w-\varepsilon)}{2\varepsilon}$, on a tiny network in `float64`.

For each selected weight, finite differences temporarily perturb that weight in both directions while leaving every other weight fixed. The numerical slope is then compared with the corresponding entry returned by `backward`.

We use a **tiny network** because each perturbation requires additional forward/backward evaluations. We use `float64` **only for this correctness check** to reduce floating-point noise in the numerical derivative; the performance experiments still use `float32`.


In [ ]:
def grad_check(seed=0, eps=1e-6):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((8, 5)); y = rng.integers(0, 3, 8)
    params = init_params([5, 7, 3], seed, dtype=np.float64)
    params = [p + 0.1 * rng.standard_normal(p.shape) if p.ndim == 1 else p for p in params]   # non-zero biases
    _, grads = loss_and_grads(np, params, X, y)
    worst = 0.0
    for p, g in zip(params, grads):
        flat = p.reshape(-1)                                    # a *view*: editing flat edits p
        for k in rng.choice(flat.size, size=min(6, flat.size), replace=False):
            old = flat[k]
            flat[k] = old + eps; lp, _ = loss_and_grads(np, params, X, y)
            flat[k] = old - eps; lm, _ = loss_and_grads(np, params, X, y)
            flat[k] = old
            num, ana = (lp - lm) / (2 * eps), g.reshape(-1)[k]
            worst = max(worst, abs(num - ana) / max(1e-12, abs(num) + abs(ana)))
    return worst

err = grad_check()
print(f"worst relative error: {err:.2e}", "→ backward() is correct ✔" if err < 1e-5 else "→ BUG!")

A small relative error gives evidence that the tested entries and shapes are correct; it is not a proof for every possible input. The result need not be exactly zero because both the analytic calculation and the finite-difference approximation involve floating-point arithmetic. A large error is a signal to inspect indexing, missing division by `B`, ReLU masking, and matrix transposes **before** optimising anything.


## B4 · The training loop (works for NumPy *and* CuPy)

Read the loop carefully, because **we will not change it for the GPU**. Notes for later:

* `Xd, yd = xp.asarray(...)`: with NumPy this is a no-op; with CuPy it is the **one upload** (rule: *in once, out once*).
* the per-epoch shuffle is created on the CPU, uploaded as **one** array, and used to slice batches on the device.
* the running loss `total` stays on the device; we convert it to a Python number **once per epoch**. (`float(loss)` every step would force a GPU→CPU round trip: a *hidden synchronisation*; we measure its cost in Part D.)
* the first few steps are a warm-up on a *copy* of the weights: CuPy compiles kernels and cuBLAS initialises on first use.

The timing boundaries matter when comparing results:

- the initial upload is reported separately as `upload_s`;
- warm-up is excluded from `epoch_s`;
- each epoch timer covers the training loop and ends after GPU synchronisation;
- test accuracy is computed **after** that timer stops, so evaluation does not inflate the training time.

Two optional arguments are used later as deliberate mistakes: `resident=False` uploads every batch, and `sync_loss_each_step=True` converts every batch loss to a Python scalar. They are there to isolate costs, not as recommended training styles.


In [ ]:
def accuracy(xp, params, X, y, chunk=5000):
    correct = 0
    for i in range(0, X.shape[0], chunk):
        pred = forward(xp, params, X[i:i+chunk])[-1].argmax(axis=1)
        correct += int((pred == y[i:i+chunk]).sum())
    return correct / X.shape[0]

def train(xp, data, hidden=(512,), batch=256, epochs=5, lr=0.1, seed=0,
          resident=True, sync_loss_each_step=False, verbose=True):
    Xtr, ytr, Xte, yte = data
    sync = sync_gpu if xp is cp else (lambda: None)
    params = [xp.asarray(p) for p in init_params([Xtr.shape[1], *hidden, 10], seed)]
    n = Xtr.shape[0]

    t0 = time.perf_counter()
    if resident or xp is np:
        Xd, yd = xp.asarray(Xtr), xp.asarray(ytr)            # ONE upload (a no-op for NumPy)
    Xted, yted = xp.asarray(Xte), xp.asarray(yte)
    sync(); upload_s = time.perf_counter() - t0

    for _ in range(3):                                       # warm-up on throw-away weights
        train_step(xp, [p.copy() for p in params], xp.asarray(Xtr[:batch]), xp.asarray(ytr[:batch]), 0.0)
    sync()

    epoch_s, losses, accs = [], [], []
    for ep in range(epochs):
        perm = np.random.default_rng(seed + ep).permutation(n)
        perm_d = xp.asarray(perm) if (resident or xp is np) else None
        total = xp.zeros((), dtype=xp.float32)
        sync(); t0 = time.perf_counter()
        for i in range(0, n - batch + 1, batch):
            if resident or xp is np:
                idx = perm_d[i:i+batch]
                xb, yb = Xd[idx], yd[idx]
            else:                                            # "streaming": gather on CPU, upload every batch
                idx = perm[i:i+batch]
                xb, yb = xp.asarray(Xtr[idx]), xp.asarray(ytr[idx])
            loss = train_step(xp, params, xb, yb, lr)
            total += loss
            if sync_loss_each_step:
                float(loss)                                  # forces a GPU→CPU round trip every step
        sync(); epoch_s.append(time.perf_counter() - t0)
        losses.append(float(total) / (n // batch))
        accs.append(accuracy(xp, params, Xted, yted))
        if verbose:
            print(f"  epoch {ep+1}/{epochs}: loss {losses[-1]:.4f} | test acc {accs[-1]*100:5.2f}% | {epoch_s[-1]:6.2f} s")
    return dict(params=params, epoch_s=epoch_s, loss=losses, acc=accs, upload_s=upload_s)

### Training-loop walkthrough

For each epoch, the loop performs the following sequence:

1. Create a permutation so batches see a reproducible shuffled order.
2. Select one batch and compute its loss and gradients.
3. Update every parameter with SGD.
4. Add the batch loss to `total`, which remains in the active array backend.
5. Synchronise before stopping the timer.
6. Read the epoch loss once, then evaluate test accuracy.

`accuracy` evaluates in chunks to avoid allocating all test activations at once. On the GPU, converting the number of correct predictions to a Python `int` does synchronise, but that happens outside the timed training loop.


### Run it on the CPU (NumPy baseline)

This is the reference for every later speedup. Keep the model, batch size, learning rate, seed, and number of epochs fixed; otherwise a faster time could come from changing the experiment rather than changing the backend. The summary stores the **median epoch time**, while the final accuracy comes from the last epoch.


In [ ]:
EPOCHS = 5
print("NumPy on CPU, MLP 784-512-10, batch 256:")
res_np = train(np, data, hidden=(512,), batch=256, epochs=EPOCHS)
results["NumPy (CPU)"] = dict(epoch_s=float(np.median(res_np["epoch_s"])), acc=res_np["acc"][-1])
print(f"→ {results['NumPy (CPU)']['epoch_s']:.2f} s per epoch, final test accuracy {res_np['acc'][-1]*100:.2f}%")

Watch the loss decrease over epochs and the test accuracy generally improve. The first epoch can differ slightly from later ones because of allocator and cache effects; using the median epoch time reduces the influence of that variation. Do not expect exactly the same numbers on another run or machine.


📝 *Write down:* NumPy time per epoch = ______ s, test accuracy = ______ %

## B5 · Week 5 callback: *split, move, wait* with gradients

Week 5's four questions: how to **split**, who does what, what **moves**, who **waits**. Here is data parallelism in six lines.
Split one batch into 4 shards, compute each shard's gradient *independently* (these could be 4 workers!), then **average** the gradients.
The result equals the gradient of the whole batch. This is exactly what DDP (Weeks 10–11) does, with an all-reduce as the "average" step.

This works because the loss is a mean and differentiation is linear. With four equally sized shards,

$$\nabla L_{\text{batch}}=\frac{1}{4}\sum_{s=1}^{4}\nabla L_s.$$

For unequal shard sizes, a plain average would be wrong: use a weighted average proportional to the number of examples in each shard.


In [ ]:
params = init_params([784, 512, 10])
xb, yb = Xtr[:256], ytr[:256]

_, full = loss_and_grads(np, params, xb, yb)
shard_grads = [loss_and_grads(np, params, xs, ys)[1]
               for xs, ys in zip(np.array_split(xb, 4), np.array_split(yb, 4))]        # 4 "workers"
avg = [sum(g[i] for g in shard_grads) / 4 for i in range(len(params))]                  # the "all-reduce"
print("max |full-batch grad − averaged shard grads| =", max(float(np.abs(f - a).max()) for f, a in zip(full, avg)))

The printed maximum difference should be zero or very close to floating-point round-off. The computation is still serial in this demonstration—the point is the **decomposition**. In distributed data-parallel training, workers compute shard gradients in parallel and an all-reduce shares and averages them.


## B6 · Week 1 callback: where does one step spend its time? (Amdahl)

Time the five matmuls alone, then the whole step. The fraction `f` of time in matmul tells us how much a *matmul-only* accelerator could help (Amdahl: speedup ≤ 1/(1−f)).

If matmuls account for fraction `f` of the original step and everything else is unchanged, Amdahl's law gives

$$S_{\max}=\frac{1}{1-f}$$

for the limiting case in which matmul time is reduced to zero. This is an **upper bound for a matmul-only improvement**, not a prediction of the measured GPU speedup. A real GPU also accelerates other operations, introduces launches and transfers, and may have a different performance profile.


In [ ]:
params = init_params([784, 512, 10]); xb, yb = Xtr[:256], ytr[:256]
W1, W2 = params[0], params[2]
h1 = np.maximum(xb @ W1, 0); d2 = np.random.rand(256, 10).astype(np.float32); d1 = np.random.rand(256, 512).astype(np.float32)

def five_matmuls():
    xb @ W1; h1 @ W2; h1.T @ d2; d2 @ W2.T; xb.T @ d1

t_mm   = timeit(five_matmuls, repeat=20)
t_step = timeit(lambda: train_step(np, params, xb, yb, 0.0), repeat=20)
f = t_mm / t_step
print(f"5 matmuls: {t_mm*1e3:.2f} ms of a {t_step*1e3:.2f} ms step → f = {f:.0%}")
print(f"Amdahl bound if ONLY matmul were infinitely fast: {1/(1-f):.1f}×")
print("→ CuPy moves *everything* to the GPU, so the element-wise 'serial' part also shrinks.")

Use `f` as a diagnostic: a large value means dense linear algebra is an important part of this step, while the remaining fraction warns that accelerating matmul alone is insufficient. The standalone five-matmul timing is an approximation, so focus on the magnitude of `f` and the Amdahl bound rather than treating them as exact constants.


---
# Part C — Meet the GPU with simple tasks (CuPy)

CuPy implements a large subset of the NumPy API. The syntax is often familiar, but execution introduces three ideas that NumPy hides:

1. **Location:** NumPy arrays normally live in CPU RAM; CuPy arrays live on a GPU.
2. **Execution:** operations on CuPy arrays launch GPU work, and kernel launches are asynchronous with respect to Python.
3. **Movement:** `cp.asarray` and `cp.asnumpy` make host/device transfers explicit.

## C1 · Hello, GPU

Arrays created with `cp.` are allocated on the **current GPU**. Their operations produce more CuPy arrays, so intermediate results stay in VRAM. NumPy and CuPy arrays should not be mixed implicitly: convert both operands to the same location first.


In [ ]:
x = cp.arange(10, dtype=cp.float32)          # lives in VRAM
y = x * 2 + 1                                # two kernels run on the GPU
print(type(y).__name__, "| shape", y.shape, "| dtype", y.dtype)
print("sum on GPU:", y.sum(), "(still a GPU scalar)")

y_cpu = cp.asnumpy(y)                         # GPU → CPU copy (also written y.get())
print("back on the CPU:", type(y_cpu).__name__, y_cpu)

### What C1 demonstrates

- `type(y)` is a CuPy array type, while `y_cpu` is a NumPy array.
- `y.sum()` is a zero-dimensional **device array**, not yet an ordinary Python float.
- `cp.asnumpy(y)` (equivalently `y.get()`) copies the values to host memory and waits for the device work needed to produce them.

A useful debugging habit is to ask of every variable: **What is its type, shape, dtype, and device?** Most accidental transfers and backend-mixing errors become obvious when you track those four properties.


**Porting rules (keep this list!)**

1. Replace `np.` with `cp.` only for supported operations; slicing, `@`, broadcasting, and reductions keep their familiar syntax.
2. Know where each array lives and do not mix NumPy and CuPy operands implicitly.
3. Upload **once** with `cp.asarray`; download **once** with `cp.asnumpy` or `.get()` when the workflow allows it.
4. Use `float32` for this T4 workload; NumPy often creates `float64` by default (Part C5).
5. **Kernel launches** usually return before execution finishes. Synchronise before stopping a wall-clock measurement.
6. Converting a device scalar to Python with `float(x)` or `.item()` waits for the GPU; doing this every step can serialise the loop.
7. Write functions as `f(xp, ...)` to reuse one implementation for NumPy and CuPy, as we did in Part B.

## C2 · Simple task 1: element-wise ReLU on 10 million numbers (and a timing trap)

ReLU performs almost no arithmetic per value. For `n` float32 values, an ideal implementation reads `4n` bytes and writes `4n` bytes, so its estimated traffic is `8n` bytes. That makes it a useful contrast with matmul: performance should be governed mainly by **memory bandwidth**.


In [ ]:
n = 10_000_000
x_cpu = np.random.standard_normal(n).astype(np.float32)
x_gpu = cp.asarray(x_cpu)
cp.maximum(x_gpu, 0); sync_gpu()                     # warm-up (first call compiles the kernel)

# ❌ WRONG: time only the *launch*
t0 = time.perf_counter(); y = cp.maximum(x_gpu, 0); t_wrong = time.perf_counter() - t0
sync_gpu()

# ✅ RIGHT: wait for the GPU before reading the clock
t_gpu = timeit(lambda: cp.maximum(x_gpu, 0), sync=sync_gpu, repeat=20)
t_cpu = timeit(lambda: np.maximum(x_cpu, 0), repeat=20)

print(f"NumPy                       : {t_cpu*1e3:8.3f} ms")
print(f"CuPy, no sync (WRONG)       : {t_wrong*1e3:8.3f} ms   ← just the time to queue the kernel!")
print(f"CuPy, with sync (correct)   : {t_gpu*1e3:8.3f} ms   → {t_cpu/t_gpu:.1f}× faster than NumPy")
gbs = 2 * 4 * n / t_gpu / 1e9
print(f"memory traffic {gbs:.0f} GB/s (T4 peak ≈ 320 GB/s): ReLU does almost no math, so it is *bandwidth*-bound")

📝 *Write down:* the wrong measurement says ______ ms, the right one ______ ms. Why is the wrong number so small?

**Kernel fusion preview.** `1 / (1 + exp(-x))` (sigmoid) launches *four* kernels and creates three temporary arrays in VRAM.
`cp.fuse` merges them into one kernel: each element is read once and written once. (In Week 8 you will write such kernels yourself.)

The incorrect measurement ends before the GPU has executed the kernel; it mostly measures Python and launch overhead. The correct measurement includes execution because `sync_gpu()` runs before the final clock reading.

The bandwidth estimate is

$$\text{GB/s}=\frac{2\times 4\times n}{t\times 10^9},$$

where the factor two represents one read and one write. This estimate assumes no extra temporary array and ignores other traffic, so treat it as a useful approximation rather than a complete hardware profile.

For sigmoid, the unfused expression creates intermediate arrays for negation, exponentiation, addition, and division. Fusion reduces kernel launches and intermediate memory traffic. It helps element-wise chains; it does not turn arbitrary Python code or an entire training loop into one kernel.


In [ ]:
def sigmoid(x):
    return 1 / (1 + cp.exp(-x))

@cp.fuse()
def sigmoid_fused(x):
    return 1 / (1 + cp.exp(-x))

sigmoid(x_gpu); sigmoid_fused(x_gpu); sync_gpu()
t1 = timeit(lambda: sigmoid(x_gpu), sync=sync_gpu, repeat=20)
t2 = timeit(lambda: sigmoid_fused(x_gpu), sync=sync_gpu, repeat=20)
t0 = timeit(lambda: 1 / (1 + np.exp(-x_cpu)), repeat=5)
print(f"NumPy {t0*1e3:7.2f} ms | CuPy 4 kernels {t1*1e3:6.3f} ms | CuPy fused {t2*1e3:6.3f} ms")

## C3 · Simple task 2: matrix multiplication, where the GPU really shines

Matmul does `2n³` FLOPs on `3n²` numbers: lots of arithmetic per byte. Three timings per size:

* **NumPy** (CPU, multi-threaded BLAS)
* **CuPy compute only** (data already on the GPU)
* **CuPy end-to-end** (upload both matrices + matmul + download the result)

📝 *Predict first:* at what `n` will the GPU win on compute-only? And end-to-end? ______

The distinction between the two GPU columns is essential:

- **Compute only** assumes the inputs already reside on the GPU and excludes the output download. It is relevant when arrays are reused in a longer device-side workflow.
- **End-to-end** includes uploading both inputs and downloading the result. It is relevant to a one-off CPU → GPU → CPU operation.

The **crossover** is the first tested size at which one implementation becomes faster. It is not a universal constant: it moves with the CPU, GPU, PCIe link, dtype, library versions, and timing method.


In [ ]:
sizes = [128, 256, 512, 1024, 2048, 4096]
rows = []
for n in sizes:
    A = np.random.rand(n, n).astype(np.float32); Bm = np.random.rand(n, n).astype(np.float32)
    Ag, Bg = cp.asarray(A), cp.asarray(Bm)
    t_cpu = timeit(lambda: A @ Bm, repeat=3 if n >= 2048 else 7)
    t_gpu = timeit(lambda: Ag @ Bg, sync=sync_gpu, repeat=20)
    t_e2e = timeit(lambda: cp.asnumpy(cp.asarray(A) @ cp.asarray(Bm)), repeat=5)
    rows.append((n, t_cpu, t_gpu, t_e2e))

print(f"{'n':>5} | {'NumPy ms':>9} | {'GPU ms':>8} | {'GPU end-to-end ms':>17} | {'speedup':>8} | {'GPU GFLOP/s':>11}")
for n, tc, tg, te in rows:
    print(f"{n:>5} | {tc*1e3:9.2f} | {tg*1e3:8.3f} | {te*1e3:17.2f} | {tc/tg:7.1f}× | {2*n**3/tg/1e9:11.0f}")

In [ ]:
ns = [r[0] for r in rows]
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].loglog(ns, [r[1] for r in rows], "o-", label="NumPy (CPU)")
axs[0].loglog(ns, [r[2] for r in rows], "o-", label="CuPy compute only")
axs[0].loglog(ns, [r[3] for r in rows], "o--", label="CuPy end-to-end")
axs[0].set_xlabel("n (matrix is n×n)"); axs[0].set_ylabel("seconds"); axs[0].legend(); axs[0].set_title("matmul time")
axs[1].semilogx(ns, [r[1]/r[2] for r in rows], "o-", label="compute only")
axs[1].semilogx(ns, [r[1]/r[3] for r in rows], "o--", label="end-to-end")
axs[1].axhline(1, color="gray", lw=1); axs[1].set_xlabel("n"); axs[1].set_ylabel("speedup over NumPy"); axs[1].legend(); axs[1].set_title("GPU speedup (above 1 = GPU wins)")
plt.tight_layout(); plt.show()

### How to read the plots

On the left, lower time is better. On the right, the horizontal line at speedup = 1 is the break-even point: above it the GPU wins; below it NumPy wins. Compare the solid compute-only curve with the dashed end-to-end curve to see the cost of transfers.

On the log–log time plot, matmul work grows roughly as `n³`, while the matrices' memory footprint grows as `n²`. As `n` increases, arithmetic therefore has more opportunity to dominate fixed launch and transfer costs.


📝 *Questions*

1. Compute-only: the GPU first wins at n = ______. End-to-end: n = ______.
2. Matmul compute grows like n³ but transfer grows like n². Why does the GPU *inevitably* win as n grows? (Think Amdahl.)
3. The "GPU GFLOP/s" column against the T4's ~8,000 peak: what percentage do we reach at n = 4096? And the CPU number from A1?

## C4 · The transfer trap: what does one trip over PCIe cost?

For question 2, “inevitably” refers to the asymptotic trend, not a promise about every finite size or machine. Compute can grow faster than transfer volume, but the actual crossover still depends on achieved rates and fixed overheads.

## C4 · The transfer trap: what does one trip over PCIe cost?

This cell measures **upload only**. Bandwidth is calculated as bytes transferred divided by measured time. For small arrays, fixed latency dominates; for larger arrays, the transfer has enough work to approach the link's sustained bandwidth.


In [ ]:
print(f"{'size':>10} | {'upload':>10} | {'bandwidth':>10}")
for mb in [0.004, 0.4, 4, 40, 400]:
    n = int(mb * 1e6 / 4)
    h = np.random.rand(n).astype(np.float32)
    t = timeit(lambda: cp.asarray(h), sync=sync_gpu, repeat=10)
    print(f"{mb:8.3f} MB | {t*1e6:7.0f} µs | {n*4/t/1e9:6.1f} GB/s")
print("\nA tiny array pays a fixed latency (≈10–50 µs) and sees almost no bandwidth: don't ping-pong small arrays.")

Notice that the smallest transfers have very low calculated GB/s even though the link itself has not changed. The same fixed cost is simply spread over fewer bytes. This is why many tiny uploads can be worse than one larger, well-organised upload—and why end-to-end GPU time must include the transfers your workflow actually performs.


**Rule of the week:** *upload once → compute a lot → download once.* Now the same idea for training data: MNIST is 188 MB, which uploads in tens of milliseconds. We do that **once**, not once per batch (we'll measure the difference in Part D).

## C5 · Why float32? One more reason

Consumer and Colab GPUs have far fewer FP64 units than FP32 units (T4: 1/32). NumPy's default dtype is `float64`: forget `.astype(np.float32)` and you silently run on the slow units.

The FP64 ratio is **hardware-specific**: the T4 is particularly weak at FP64 compared with FP32, while some datacenter GPUs have much stronger FP64 support. Independently of arithmetic speed, `float64` also doubles the bytes of each array, increasing memory footprint and memory traffic.

Compare both reported time and achieved TFLOP/s. The lesson is not “never use float64”; it is **choose the dtype deliberately for the hardware and numerical requirements**, and do not inherit NumPy's default accidentally.


In [ ]:
n = 4096
for dt in (cp.float32, cp.float64):
    a = cp.random.standard_normal((n, n), dtype=dt)
    t = timeit(lambda: a @ a, sync=sync_gpu, repeat=5)
    print(f"{cp.dtype(dt).name:>8}: {t*1e3:8.1f} ms   {2*n**3/t/1e12:5.2f} TFLOP/s")

---
# Part D — Train the *same* MLP on the GPU

Now the payoff: call `train()` with `xp = cp`. **Not one line of the model or the loop changes.**

The comparison uses the same initial parameter seed and the same per-epoch shuffling logic. That makes the training procedure closely matched, but it does not guarantee bit-identical execution: CPU and GPU libraries can tile operations and reduce floating-point values in different orders.

Before running, predict which will be faster and whether the accuracy should be **similar** or **exactly identical**.


In [ ]:
print("CuPy on GPU, MLP 784-512-10, batch 256:")
res_cp = train(cp, data, hidden=(512,), batch=256, epochs=EPOCHS)
results["CuPy (GPU)"] = dict(epoch_s=float(np.median(res_cp["epoch_s"])), acc=res_cp["acc"][-1])

sp = results["NumPy (CPU)"]["epoch_s"] / results["CuPy (GPU)"]["epoch_s"]
print(f"→ {results['CuPy (GPU)']['epoch_s']:.3f} s per epoch (NumPy: {results['NumPy (CPU)']['epoch_s']:.2f} s) → {sp:.1f}× faster; "
      f"one-time upload {res_cp['upload_s']*1e3:.0f} ms; test accuracy {res_cp['acc'][-1]*100:.2f}%")

diff = max(float(np.abs(cp.asnumpy(a) - b).max()) for a, b in zip(res_cp["params"], res_np["params"]))
print(f"max |weight difference| GPU vs CPU after {EPOCHS} epochs: {diff:.2e}  (same maths, different order of float additions)")

### How to read the first GPU training result

- **Seconds per epoch** is the steady-state training time; the one-time upload is printed separately.
- **Speedup** is NumPy time divided by CuPy time using the same epochs and batch size.
- **Test accuracy** should be broadly similar, though small differences are normal.
- **Maximum weight difference** checks numerical divergence after training. A nonzero value does not by itself indicate a bug; floating-point addition is not associative, and the backends do not execute every operation in exactly the same order.

If accuracy or loss differs dramatically, investigate correctness and hyperparameters rather than explaining it away as ordinary rounding.


📝 *Write down:* GPU time per epoch = ______ s, speedup = ______×, test accuracy = ______ %

*Why isn't the speedup closer to what the peak numbers promise?* A T4 can do several TFLOP/s but our step is only 0.6 GFLOP, i.e. roughly **0.1–0.2 ms of real math**.
Each step launches dozens of small kernels with ~10 µs of launch overhead each, so a model this small cannot keep the GPU busy. The sweep below tests this explanation.

## D1 · When does the GPU win? A sweep over model size and batch size

We time a single training step (no data loading) for a grid of hidden widths `H` and batch sizes `B`.

The sweep isolates computation: data are already on the appropriate device, and the learning rate is zero because we are timing a step rather than evaluating learning. Increasing `B` adds more rows to each activation matrix; increasing `H` adds more outputs and arithmetic to the hidden layer.

📝 **Predict the pattern first:** which corner should be most launch-bound, and which should give the GPU the most useful work per launch?


In [ ]:
Hs = [64, 256, 1024, 4096]
Bs = [32, 128, 512, 2048]
speed = np.zeros((len(Bs), len(Hs)))
t_cpu_grid = np.zeros_like(speed); t_gpu_grid = np.zeros_like(speed)
for i, B in enumerate(Bs):
    for j, H in enumerate(Hs):
        p0 = init_params([784, H, 10])
        xb = np.random.standard_normal((B, 784)).astype(np.float32); yb = np.random.randint(0, 10, B)
        pc = [p.copy() for p in p0]
        t_c = timeit(lambda: train_step(np, pc, xb, yb, 0.0), repeat=5)
        pg = [cp.asarray(p) for p in p0]; xg, yg = cp.asarray(xb), cp.asarray(yb)
        t_g = timeit(lambda: train_step(cp, pg, xg, yg, 0.0), sync=sync_gpu, repeat=20)
        t_cpu_grid[i, j], t_gpu_grid[i, j], speed[i, j] = t_c, t_g, t_c / t_g

fig, ax = plt.subplots(figsize=(6.4, 4.2))
im = ax.imshow(np.log10(speed), cmap="RdYlGn", vmin=-1, vmax=2)
ax.set_xticks(range(len(Hs)), [f"H={h}" for h in Hs]); ax.set_yticks(range(len(Bs)), [f"B={b}" for b in Bs])
for i in range(len(Bs)):
    for j in range(len(Hs)):
        ax.text(j, i, f"{speed[i, j]:.1f}×", ha="center", va="center", fontsize=11)
ax.set_title("GPU speedup over NumPy, one training step\n(green: GPU wins, red: CPU wins)"); plt.tight_layout(); plt.show()

Each cell reports `CPU time / GPU time`, so **greater than 1 means the GPU wins**. Read across a row to see the effect of hidden width, and down a column to see the effect of batch size. The colours use a logarithmic scale so that slowdowns and speedups are both visible.

This grid measures one step, not accuracy or time to convergence. A large batch can improve hardware utilisation while changing optimisation behaviour, so hardware efficiency and training quality must be evaluated separately.


📝 *Questions*

1. In which corner of the grid does the CPU win or tie? Explain with *kernel launch overhead*.
2. Which corner has the biggest speedup? What does a larger batch or width do to the work per kernel?
3. Practical advice for a student training a tiny model: is a GPU always worth it?

## D2 · Three ways to throw the speedup away

Same model, same GPU, one epoch each. Only the *style* changes.

For question 3, include the full context: model size, batch size, transfer frequency, CPU quality, and whether the data can remain resident. “GPU available” does not automatically imply “GPU beneficial”.

## D2 · Three ways to throw the speedup away

This is a controlled comparison: the model and data stay fixed while one loop decision changes at a time. Run it more than once if the numbers are noisy, and focus on the **relative pattern**. The NumPy row is a reference, not one of the three GPU configurations.


In [ ]:
cfgs = [("baseline: data on GPU, loss synced once/epoch", dict()),
        ("hidden sync: float(loss) every step",           dict(sync_loss_each_step=True)),
        ("transfer trap: upload every batch from CPU",    dict(resident=False))]
for label, kw in cfgs:
    r = train(cp, data, hidden=(512,), batch=256, epochs=1, verbose=False, **kw)
    print(f"{label:<50}: {r['epoch_s'][0]:6.3f} s/epoch")
r = train(np, data, hidden=(512,), batch=256, epochs=1, verbose=False)
print(f"{'(reference) NumPy on CPU':<50}: {r['epoch_s'][0]:6.3f} s/epoch")

**Take-away.** The GPU gives its speedup only if the loop stays *on* the device: no per-step `float(loss)`, no per-batch uploads.
(In Week 9 you will see how `DataLoader`, `pin_memory` and `non_blocking` hide the unavoidable transfers.)

## D3 · GPU memory: the other limit

CuPy keeps freed blocks in a **memory pool** to avoid slow `cudaMalloc` calls. `nvidia-smi` therefore shows more than your live arrays.

`used_bytes()` is memory currently associated with live allocations from the pool. `total_bytes()` also includes cached blocks that CuPy is retaining for reuse. Therefore, `total ≥ used` is normal and does not by itself indicate a leak.

Calling `free_all_blocks()` releases unused cached blocks back to the driver; it does **not** delete live arrays. Real training still needs enough memory for parameters, gradients, activations, temporary arrays, and library workspaces.


In [ ]:
pool = cp.get_default_memory_pool()
print(f"in use by live arrays: {pool.used_bytes()/2**20:7.1f} MiB | held by the pool: {pool.total_bytes()/2**20:7.1f} MiB")
pool.free_all_blocks()
print(f"after free_all_blocks : {pool.used_bytes()/2**20:7.1f} MiB | held by the pool: {pool.total_bytes()/2**20:7.1f} MiB")
print("Memory per step ≈ weights + activations (B×H floats per layer) + gradients. Big batches or models hit 'out of memory' first, not 'too slow'.")

If you encounter CUDA out-of-memory in a later exercise, reduce batch size or hidden width first, then rerun from a clean runtime if necessary. Use the pool statistics to distinguish retained cache from live allocations. Avoid calling `free_all_blocks()` inside the training loop as a routine optimisation—it can make later allocations more expensive.


---
# Part E — The same model in PyTorch

CuPy is "NumPy on the GPU": it has no autograd, so we wrote the backward pass by hand. PyTorch
gives the same GPU arrays (tensors) **plus automatic differentiation**. From Week 9 on we use PyTorch for real training.

## E1 · Does autograd compute *our* gradients?

We build the same network, copy our initial weights in, and compare the gradients on one batch.

A PyTorch tensor plays a role similar to an array, but it can also record the operations needed for automatic differentiation. `nn.Linear` stores weights with shape `(out_features, in_features)`, which is the transpose of the `W` layout used in our matrix formulas.

`F.cross_entropy` combines the stable softmax-related calculation and classification loss. Calling `.backward()` constructs the parameter gradients automatically; it does not remove the underlying matmuls or other GPU work.


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CPU threads:", torch.get_num_threads())

def make_torch_model(hidden, seed=0):
    sizes = [784, *hidden, 10]
    layers = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        layers += [nn.Linear(a, b), nn.ReLU()]
    model = nn.Sequential(*layers[:-1])                      # no ReLU after the last layer
    params = init_params(sizes, seed)                        # identical start to the NumPy/CuPy runs
    with torch.no_grad():
        for lin, (W, b) in zip([m for m in model if isinstance(m, nn.Linear)], zip(params[0::2], params[1::2])):
            lin.weight.copy_(torch.from_numpy(W.T.copy())); lin.bias.copy_(torch.from_numpy(b))
    return model

model = make_torch_model((512,))
xb, yb = Xtr[:256], ytr[:256]
F.cross_entropy(model(torch.from_numpy(xb)), torch.from_numpy(yb)).backward()

_, ours = loss_and_grads(np, init_params([784, 512, 10]), xb, yb)
lins = [m for m in model if isinstance(m, nn.Linear)]
worst = max(float(np.abs(lins[0].weight.grad.numpy().T - ours[0]).max()), float(np.abs(lins[1].weight.grad.numpy().T - ours[2]).max()))
print(f"max |autograd − hand-written gradient| = {worst:.2e}  → autograd computes exactly the matrices we derived ✔")

The code explicitly copies our NumPy initial weights into PyTorch, transposing each weight matrix to match PyTorch's storage convention. It then compares autograd's gradients with our hand-written gradients on the same batch.

The small difference is a numerical tolerance issue, not a different mathematical gradient. Once this check passes, you can use autograd to avoid derivative bookkeeping while retaining the same model and optimisation algorithm.


## E2 · Train it on CPU and GPU with PyTorch

The training loop mirrors the CuPy version deliberately:

- move the model and data to the selected device once;
- compute forward and loss for each batch;
- clear old gradients, call `backward()`, and apply SGD with `opt.step()`;
- accumulate `loss.detach()` so logging does not retain the autograd graph;
- evaluate under `torch.no_grad()` so evaluation does not build a graph.

The same seed, model, data order, and hyperparameters make the comparison meaningful. CPU and GPU timings are still allowed to differ from CuPy because framework dispatch, memory allocation, and library calls are not identical.


In [ ]:
def train_torch(device, data, hidden=(512,), batch=256, epochs=5, lr=0.1, seed=0, verbose=True):
    Xtr, ytr, Xte, yte = data
    sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
    model = make_torch_model(hidden, seed).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    Xd, yd = torch.from_numpy(Xtr).to(device), torch.from_numpy(ytr).to(device)       # ONE upload
    Xted, yted = torch.from_numpy(Xte).to(device), torch.from_numpy(yte).to(device)
    n = Xd.shape[0]

    for _ in range(3):                                      # warm-up (cuBLAS init, allocator)
        F.cross_entropy(model(Xd[:batch]), yd[:batch]).backward()
    model.zero_grad(set_to_none=True); sync()

    epoch_s, accs = [], []
    for ep in range(epochs):
        perm = torch.from_numpy(np.random.default_rng(seed + ep).permutation(n)).to(device)
        total = torch.zeros((), device=device)
        sync(); t0 = time.perf_counter()
        for i in range(0, n - batch + 1, batch):
            idx = perm[i:i+batch]
            loss = F.cross_entropy(model(Xd[idx]), yd[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()                                  # autograd = our backward()
            opt.step()                                       # SGD update
            total += loss.detach()
        sync(); epoch_s.append(time.perf_counter() - t0)
        with torch.no_grad():
            accs.append(float((model(Xted).argmax(1) == yted).float().mean()))
        if verbose:
            print(f"  epoch {ep+1}/{epochs}: loss {float(total)/(n//batch):.4f} | test acc {accs[-1]*100:5.2f}% | {epoch_s[-1]:6.2f} s")
    return dict(epoch_s=epoch_s, acc=accs)

print("PyTorch on CPU:")
r_tc = train_torch(torch.device("cpu"), data, epochs=EPOCHS)
results["PyTorch (CPU)"] = dict(epoch_s=float(np.median(r_tc["epoch_s"])), acc=r_tc["acc"][-1])
print("PyTorch on GPU:")
r_tg = train_torch(torch.device("cuda"), data, epochs=EPOCHS)
results["PyTorch (GPU)"] = dict(epoch_s=float(np.median(r_tg["epoch_s"])), acc=r_tg["acc"][-1])

For this small model, PyTorch and CuPy may have similar epoch times because both are dominated by launching many small operations. Autograd adds bookkeeping, but it does not necessarily dominate the runtime. With larger models, PyTorch also provides convenient mixed precision, optimised layers, compilation options, data pipelines, and distributed training.


## E3 · Mixed-precision preview (Week 12): Tensor Cores

The T4 has Tensor Cores that multiply **float16** matrices much faster than the normal FP32 cores. One matmul, two precisions:

This cell benchmarks **one matmul in two precisions**; it is not yet a complete mixed-precision training recipe. Float16 reduces memory traffic and can use Tensor Cores efficiently, while real training may keep selected computations or master weights in higher precision for stability.

When comparing the result with Part C, remember that this is PyTorch on the same GPU, not a different hardware device. Differences in library heuristics and measurement conditions still matter.


In [ ]:
n = 4096
a32 = torch.randn(n, n, device="cuda"); a16 = a32.half()
for name, a in [("float32", a32), ("float16 (Tensor Cores)", a16)]:
    t = timeit(lambda: a @ a, sync=torch.cuda.synchronize, repeat=10)
    print(f"{name:>22}: {t*1e3:7.2f} ms  → {2*n**3/t/1e12:5.1f} TFLOP/s")
print("Lower precision = fewer bytes to move AND faster math. The price is accuracy care (loss scaling): Week 12.")

The expected pattern is substantially higher matmul throughput for float16 on a T4. Do not infer that every training workload will receive the same speedup: element-wise operations, memory transfers, launch overhead, numerical-stability safeguards, and small matrix dimensions can all reduce the benefit.


---
# Part F — Wrap-up

Use the table for exact recorded values and the bar chart for the overall pattern. The time axis is logarithmic, so equal visual distances represent equal **ratios**, not equal differences in seconds. Lower time is better; accuracy should be interpreted alongside time rather than assuming the fastest run is automatically the best-trained model.


In [ ]:
print(f"{'backend':<16} | {'s / epoch':>10} | {'vs NumPy':>9} | {'test acc':>8}")
base = results["NumPy (CPU)"]["epoch_s"]
for k, v in results.items():
    print(f"{k:<16} | {v['epoch_s']:10.3f} | {base/v['epoch_s']:8.1f}× | {v['acc']*100:7.2f}%")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ks = list(results); vals = [results[k]["epoch_s"] for k in ks]
bars = ax.barh(ks, vals, color=["#6b7280", "#2f6fb0", "#e08a1e", "#16a34a"][:len(ks)])
ax.set_xscale("log"); ax.set_xlabel("seconds per epoch (log scale, lower is better)"); ax.invert_yaxis()
for b, v in zip(bars, vals): ax.text(v, b.get_y() + b.get_height()/2, f" {v:.2f}s", va="center")
plt.tight_layout(); plt.show()

## 📝 Your wrap-up (submit with your results)

1. Your completed tables: A1 CPU GFLOP/s, B4 NumPy epoch time, C3 crossover size, D epoch time and speedup.
2. One sentence: *"On a T4, training this MLP on the GPU is worthwhile when … because …"* (use numbers from the sweep in D1).
3. The four course questions for the GPU: what is its answer to **split**, **assign**, **move**, **wait**? (Hints: one thread per element · the hardware block scheduler · `asarray`/`asnumpy` over PCIe · asynchronous launches + `synchronize`.)
4. PyTorch vs CuPy on this model gave similar epoch times. Why? What would PyTorch do better on a bigger model? (Hint: autograd bookkeeping vs launch overhead; mixed precision; `torch.compile`.)

## 🔧 Exercises (pick two)

* **E-1 Deeper network.** Train `hidden=(512, 256)`. The code already supports it. Is the GPU speedup larger or smaller? Why?
* **E-2 Batch size.** Train with `batch=64` and `batch=2048` on the GPU. Plot time per epoch and final accuracy. What is the trade-off?
* **E-3 Momentum.** Add SGD with momentum (keep one velocity array per parameter, using in-place updates). Momentum changes the **update**, not `loss_and_grads`, so the existing gradient check should still pass. Compare loss and accuracy after the same number of epochs; do not expect momentum to reduce raw seconds per epoch, because it adds a small update cost.
* **E-4 Float16 in CuPy.** Cast the weights and data to `cp.float16`. Does the loss still go down? Compare time per epoch. (Hint: you may need a smaller learning rate.)
* **E-5 Honest benchmark.** Compare against a CPU with more cores (your laptop?). How does the speedup change when NumPy gets 8 cores instead of 2?

---
*Next week: midterm (Weeks 1–6). Week 8: write your own CUDA kernels with Numba.*

### A good final explanation

A strong answer connects **measurement → mechanism → decision**. For example:

> “For this batch and width, the measured GPU speedup was … . The sweep suggests that … dominates at small sizes, while larger matrices provide more work per launch. Keeping data resident and avoiding per-step scalar reads matters because … . On a different CPU/GPU, I would rerun the crossover and training benchmarks before choosing a backend.”

Avoid conclusions based only on peak TFLOP/s, core counts, or one noisy repetition.


## Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `No GPU visible` or `nvidia-smi` fails | wrong runtime | Select a T4 GPU, then use **Restart and run all** so every cell uses the new runtime. |
| MNIST download fails | network restriction | The synthetic fallback preserves the code experiments, but not MNIST accuracy; rerun with network access for the intended results. |
| Every GPU operation appears to take microseconds | missing synchronisation | Use `sync_gpu()` before stopping the timer, or use the supplied `timeit(..., sync=sync_gpu)`. |
| GPU is slower than expected | small workload, transfers, scalar reads, or accidental float64 | Check data location, `resident`, `sync_loss_each_step`, dtypes, batch size, and model width. |
| Results differ slightly each run | timing noise and floating-point execution order | Warm up, repeat, use medians, and compare trends; keep seeds fixed for a controlled comparison. |
| CUDA out-of-memory | arrays/activations no longer fit | Reduce batch size or width; inspect the memory pool and restart the runtime if old allocations are no longer needed. |
| A later cell reports an undefined name | cells run out of order | Restart and run all, or rerun the earlier definition cells in order. |

> **Final check before submitting:** state your hardware, data source (MNIST or synthetic), and any settings you changed. A number without its experimental context is difficult to interpret.
